In [29]:
import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
text="aaabaa-kra rkxassaab"
print(list(text.encode("utf-8")))

[97, 97, 97, 98, 97, 97, 45, 107, 114, 97, 32, 114, 107, 120, 97, 115, 115, 97, 97, 98]


In [14]:
def count_pairs(ids, counts=None):
  if counts is None:
    counts=defaultdict(int)
  for pair in zip(ids,ids[1:]):
    counts[pair]+=1

  return counts

In [6]:
print(count_pairs(list(text.encode("utf-8"))))

defaultdict(<class 'int'>, {(97, 97): 4, (97, 98): 2, (98, 97): 1, (97, 45): 1, (45, 107): 1, (107, 114): 1, (114, 97): 1, (97, 32): 1, (32, 114): 1, (114, 107): 1, (107, 120): 1, (120, 97): 1, (97, 115): 1, (115, 115): 1, (115, 97): 1})


In [15]:
def merge(ids,pair,new_id):
  merge_ids=[]
  i=0

  while i<len(ids):
    if i<len(ids)-1 and (ids[i], ids[i+1]) == pair:
      merge_ids.append(new_id)
      i+=2
    else:
      merge_ids.append(ids[i])
      i+=1

  return merge_ids

In [31]:
def pretokenize(text):
  pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
  return re.findall(pattern,text)

'(?:[sdmt]|ll|ve|re) : 영어 축약형 ('s, 'd, 'm, 't,  'll, 've, 're)

?\p{L}+ : 단어 (앞에 공백 가능)

?\p{N}+ : 연속된 숫자

?[^\s\p{L}\p{N}]+ : 구두점 및 기호

\s+(?!\S)|\s+ : 공백 문자


findall(pattern,text) (text에서 pattern에 해당하는 부분 다 찾기)

In [32]:
def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):

  ids_list=[]
  for text in input_text.split(end_token):
    for pretoken in pretokenize(text):
      ids_list.append(list(pretoken.encode("utf-8")))

  num_merges=vocab_size-256-1
  merge_rules={}

  for step in tqdm(range(num_merges),desc="Training BPE"):
    counts=defaultdict(int)

    for ids in ids_list:
      counts= count_pairs(ids,counts)

    if not counts:
      break

    best_pair=max(counts,key=counts.get)
    new_id=step+256

    merge_rules[best_pair]=new_id

    for i in range(len(ids_list)):
      ids_list[i]=merge(ids_list[i],best_pair,new_id)

  return merge_rules

In [22]:
txt="hello world!<|endoftoken|>Nice to meet you"

merge_rules=train_bpe(txt,vocab_size=260)
print(merge_rules)

{(111, 32): 256, (101, 110): 257, (104, 101): 258}


In [10]:
print(bytes([11])+bytes([16]))

b'\x0b\x10'


In [30]:
class BPETokenizer:
  def __init__(self, merge_rules,end_token="<|endoftext|>"):
    self.merge_rules=merge_rules
    self.end_token=end_token
    self.end_token_id=256+len(merge_rules)

    self.id_to_bytes = {i: bytes([i]) for i in range(256)}

    for (id1,id2), new_id in merge_rules.items():
      self.id_to_bytes[new_id]=self.id_to_bytes[id1]+self.id_to_bytes[id2]

      self.id_to_bytes[self.end_token_id]=self.end_token.encode("utf-8")
    self.vocab_size=len(self.id_to_bytes)

  def _encode_text(self,text):
    ids=list(text.encode("utf-8"))

    for merge_pair, new_id in self.merge_rules.items():
      ids=merge(ids,merge_pair,new_id)

    return ids

  def encode(self,input_text,show_progress=False):
    pattern="("+re.escape(self.end_token)+")"
    texts=re.split(pattern,input_text)
    all_ids=[]

    texts=tqdm(texts, desc="Encoding") if show_progress else texts

    for text in texts:
      if text==self.end_token:
        all_ids.append(self.end_token_id)
      else:
        for pretoken in pretokenize(text):
          all_ids.extend(self._encode_text(pretoken))

    return all_ids

  def decode(self,ids):
    byte_list=[self.id_to_bytes[i] for i in ids]
    text_bytes=b"".join(byte_list)

    return text_bytes.decode("utf-8",errors="replace")

re.split(pattern,input) 에서 pattern에다가 괄호 씌워야 end_token도 같이 text로 넣을 수 있음. 안 그러면 end_token 빼고 split 됨

extend는 append에서 list 껍데기를 빼고 이어붙이는거라 보면 됨

In [28]:
tokenizer=BPETokenizer(merge_rules)
ids=tokenizer.encode(txt)

print(ids)
print(tokenizer.decode(ids))

[258, 108, 108, 256, 119, 111, 114, 108, 100, 33, 60, 124, 257, 100, 111, 102, 116, 111, 107, 257, 124, 62, 78, 105, 99, 101, 32, 116, 256, 109, 101, 101, 116, 32, 121, 111, 117]
hello world!<|endoftoken|>Nice to meet you


In [34]:
sample_text="Oh, and in case I don't see you.<|endoftext|>Good afternoon, Good evening and Good night."

merge_rules= train_bpe(sample_text, vocab_size=280)
tokenizer=BPETokenizer(merge_rules)

text="Good afternoon, Good evening and Good night."
ids=tokenizer.encode(text)
decoded=tokenizer.decode(ids)

print(ids)
print(decoded)

for token_id in ids:
  decoded_token=tokenizer.decode([token_id])
  print(f"{token_id} -> '{decoded_token}'")

Training BPE:   0%|          | 0/23 [00:00<?, ?it/s]

[259, 257, 102, 116, 101, 114, 110, 256, 110, 44, 264, 32, 101, 118, 101, 110, 262, 103, 261, 264, 32, 110, 105, 103, 104, 116, 46]
Good afternoon, Good evening and Good night.
259 -> 'Good'
257 -> ' a'
102 -> 'f'
116 -> 't'
101 -> 'e'
114 -> 'r'
110 -> 'n'
256 -> 'oo'
110 -> 'n'
44 -> ','
264 -> ' Good'
32 -> ' '
101 -> 'e'
118 -> 'v'
101 -> 'e'
110 -> 'n'
262 -> 'in'
103 -> 'g'
261 -> ' and'
264 -> ' Good'
32 -> ' '
110 -> 'n'
105 -> 'i'
103 -> 'g'
104 -> 'h'
116 -> 't'
46 -> '.'
